# Who is a farmer?

### An interactive website examining the representation gap between the demographics of US land owners vs. local populations at the county level.

The data is from:

* [United States Department of Agriculture National Agricultural Statistics Survey](https://quickstats.nass.usda.gov/)

* [United States Census Bureau American Community Survey 5-Year Data](https://www.census.gov/data/developers/data-sets/acs-5year/2022.html)

* [United States Census Bureau Cartographic Boundary](https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html)

Both datasets are for the year 2022.

The NASS data is made through a variety of surveys, satellite imagery, and field data.

It only includes places producing and selling (or normally selling) $1,000 or more of agricultural products annually. Both hired and unpaid family workers are counted, but contract labor is generally excluded.

The participation/responses to the survey have declined over the years and data is supressed/redacted to avoid presonally identifiable data. Both of these can skew the results.

The Census data is made through a variety of surveys and community outreach.

Only residents who have lived, or intend to live, in the sampled unit for more than two months are counted. Short-term and international residents are not counted. Data is supressed/redacted to avoid personally identifiable data. Non-residential addresses are excluded.

The participation/respones to the surveys vary greatly and have generally declined and smaller geographic areas have a larger margin of error. These together potentially introduce significant error to the data we have.

# United States Department of Agriculture (USDA) National Agricultural Statistics Survey (NASS) data clean-up

### NASS Farm Acre ownership demographics

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import unicodedata

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

DATA_DIR = "./data"
OUTPUT_DIR = "./outputs/modified_data"

# Ensure output subfolder exists programmatically so exports never fail
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. NASS DEMOGRAPHIC DATA TRANSFORMATION
# ==========================================

def transform_nass_demographics(df):
    """
    Filters raw NASS survey data for acreage metrics, pivots the table from long 
    to wide format by demographic class, and estimates unspecified Asian subgroups.
    """
    print("Filtering and cleaning demographic data...")

    # Filter for acre units and unspecified categorical domains, then explicitly 
    # create a copy to avoid SettingWithCopyWarning during subsequent modifications
    filtered_df = df[
        (df['unit_desc'] == 'ACRES') &
        (df['domaincat_desc'] == 'NOT SPECIFIED')
    ].copy()

    # Clean the 'Value' column by removing commas and converting to numeric type
    filtered_df['Value'] = pd.to_numeric(
        filtered_df['Value'].astype(str).str.replace(',', ''), 
        errors='coerce'
    )

    print("Pivoting table to wide format...")
    # Pivot so each demographic class becomes its own column per county
    pivot_df = filtered_df.pivot_table(
        index=['state_alpha', 'state_name', 'county_name'],
        columns='class_desc',
        values='Value',
        aggfunc='sum'
    ).reset_index()

    # Estimate 'ASIAN, UNSPECIFIED' by subtracting known specific subgroups from the total ASIAN category
    asian_subgroups = [
        'ASIAN, CHINESE', 'ASIAN, FILIPINO', 'ASIAN, JAPANESE',
        'ASIAN, KOREAN', 'ASIAN, OTHER'
    ]

    existing_subgroups = [col for col in asian_subgroups if col in pivot_df.columns]

    if 'ASIAN' in pivot_df.columns:
        pivot_df['ASIAN, UNSPECIFIED'] = (
            pivot_df['ASIAN'].fillna(0) - pivot_df[existing_subgroups].fillna(0).sum(axis=1)
        )
        pivot_df['ASIAN, UNSPECIFIED'] = pivot_df['ASIAN, UNSPECIFIED'].clip(lower=0)

    return pivot_df


# ==========================================
# 2. DATA LOADING & MERGING
# ==========================================

# Load raw NASS universal data export and apply transformation function using relative path structure
nass_csv_path = os.path.join(DATA_DIR, "nass_universal_export.csv")
df = pd.read_csv(nass_csv_path)
pivot_df = transform_nass_demographics(df)

# Load physical total acres baseline dataset for demographic percentage calculations
totals_csv_path = os.path.join(DATA_DIR, 'nass_total_physical_acres_2022.csv')
df_totals = pd.read_csv(totals_csv_path)

print("Merging demographics with Physical Total Acres...")

# Merge demographic data with total physical acres using a left join 
# to preserve all counties even if totals are missing
master_df = pd.merge(
    pivot_df,
    df_totals,
    on=['state_alpha', 'county_name'],
    how='left'
)


# ==========================================
# 3. PERCENTAGE CALCULATIONS
# ==========================================

# Define demographic target columns to calculate proportion of total physical land
target_columns = [
    'WHITE',
    'BLACK OR AFRICAN AMERICAN',
    'AMERICAN INDIAN OR ALASKA NATIVE',
    'ASIAN',
    'ASIAN, CHINESE',
    'ASIAN, FILIPINO',
    'ASIAN, JAPANESE',
    'ASIAN, KOREAN',
    'ASIAN, OTHER',
    'ASIAN, UNSPECIFIED',
    'NATIVE HAWAIIAN',
    'PACIFIC ISLANDER, (EXCL NATIVE HAWAIIAN)',
    'MULTI-RACE',
    'HISPANIC',
    'MALE',
    'FEMALE'
]

print("Calculating percentages...")

# Compute percentage relative to true physical total acres per county
for col in target_columns:
    if col in master_df.columns:
        pct_col_name = f"{col} (% of Total Acres)"

        # Calculate percentage (results in NaN if Total_Physical_Acres is 0 or missing)
        master_df[pct_col_name] = (master_df[col] / master_df['Total_Physical_Acres']) * 100

        # Round to 2 decimal places for publication-clean numbers
        master_df[pct_col_name] = master_df[pct_col_name].round(2)


# ==========================================
# 4. CLEANUP & COLUMN FORMATTING
# ==========================================

print("Dropping redundant and overlapping columns...")

# Remove unmapped columns, aggregate classes, and overlapping 'alone or combined' categories
columns_to_drop = [
    'ALL CLASSES',
    # Unused metrics for this specific analysis scope
    'AGE LT 35',
    'MILITARY SERVICE, ACTIVE DUTY NOW OR IN THE PAST',
    # Overlapping multi-count demographic categories
    'AMERICAN INDIAN OR ALASKA NATIVE, ALONE OR COMBINED WITH OTHER RACES',
    'ASIAN, ALONE OR COMBINED WITH OTHER RACES',
    'BLACK OR AFRICAN AMERICAN, ALONE OR COMBINED WITH OTHER RACES',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER, ALONE OR COMBINED WITH OTHER RACES',
    'WHITE, ALONE OR COMBINED WITH OTHER RACES',
]

master_df = master_df.drop(columns=columns_to_drop, errors='ignore')

print("Renaming NASS demographic columns...")

# Explicitly append " (Farm Acres)" suffix to raw demographic counts for clarity, 
# leaving geographic identifiers, totals, and percentages untouched.
rename_nass = {}
for col in master_df.columns:
    if col not in ['state_name', 'state_alpha', 'county_name', 'Total_Physical_Acres'] and "(%" not in col:
        rename_nass[col] = f"{col} (Farm Acres)"

master_df = master_df.rename(columns=rename_nass)

print("Final NASS dataset complete!")

# Export master dataset to outputs/modified_data directory for subsequent mapping and analysis
output_file_path = os.path.join(OUTPUT_DIR, "Final_NASS_Data.csv")
master_df.to_csv(output_file_path, index=False)
print(f"Saved successfully to {output_file_path}")

Filtering and cleaning demographic data...
Pivoting table to wide format...
Merging demographics with Physical Total Acres...
Calculating percentages...
Dropping redundant and overlapping columns...
Renaming NASS demographic columns...
Final NASS dataset complete!
Saved successfully to ./outputs/modified_data/Final_NASS_Data.csv


# CENSUS DATA CLEANING
### Split Census data into demographics and income

In [2]:
import os
import numpy as np
import pandas as pd

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

DATA_DIR = "./data"
OUTPUT_DIR = "./outputs/modified_data"

# Ensure output subfolder exists programmatically so exports never fail
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. CENSUS GEOGRAPHY FORMATTING FUNCTIONS
# ==========================================

def extract_state_county(name_str):
    """
    Parses standard Census 'NAME' strings (e.g., 'Autauga County, Alabama') 
    into standardized state and county names to match NASS formatting.
    """
    parts = name_str.split(',')
    if len(parts) == 2:
        county_part = parts[0].strip()
        state_part = parts[1].strip()

        # Convert state name to uppercase for matching
        state_name = state_part.upper()

        # Clean county name by converting to uppercase and stripping administrative suffixes
        county_name = (
            county_part.upper()
            .replace(' COUNTY', '')
            .replace(' PARISH', '')
            .replace(' BOROUGH', '')
            .replace(' CENSUS AREA', '')
        )

        return pd.Series([state_name, county_name])
    return pd.Series([None, None])


def format_census_geography(df):
    """
    Applies geography extraction to a DataFrame and prepends the standardized 
    state and county name columns.
    """
    new_cols = df['NAME'].apply(extract_state_county)
    new_cols.columns = ['state_name', 'county_name']
    df_final = pd.concat([new_cols, df], axis=1)

    return df_final


# ==========================================
# 2. DATA LOADING & COLUMN FILTERING
# ==========================================

print("Loading raw Census dataset...")
census_csv_path = os.path.join(DATA_DIR, 'full_census_data_2022.csv')
census_df = pd.read_csv(census_csv_path)

# Define base geographic identifier columns required in split datasets
base_cols = ['state', 'county', 'NAME']

# Identify income-related columns via case-insensitive keyword search for 'income' or 'dollars'
income_cols = [
    col for col in census_df.columns 
    if col not in base_cols and ('income' in col.lower() or 'dollars' in col.lower())
]

# Identify demographic columns as everything remaining outside base and income groups
demo_cols = [
    col for col in census_df.columns 
    if col not in base_cols and col not in income_cols
]

print("Splitting into demographics and income subsets...")
# Create independent dataframes for demographics and income
df_income = census_df[base_cols + income_cols].copy()
df_demographics = census_df[base_cols + demo_cols].copy()


# ==========================================
# 3. CLEANING CENSUS MISSING VALUE CODES
# ==========================================

print("Cleaning special Census missing/withheld value codes...")

# Standard Census missing value error codes (covering both integer and float representations):
# -666666666: Data not available/withheld
# -888888888: Data not available
# -999999999: Data not applicable
# -222222222: Value too small
# -333333333: Geographies with fewer than a certain threshold of cases
replace_values = [
    -666666666, -888888888, -999999999, -222222222, -333333333,
    -666666666.0, -888888888.0, -999999999.0, -222222222.0, -333333333.0
]

# Apply replacement to numeric columns in the income dataset
numeric_cols = df_income.select_dtypes(include=[np.number]).columns

for val in replace_values:
    df_income[numeric_cols] = df_income[numeric_cols].replace(val, np.nan)


# ==========================================
# 4. FORMAT GEOGRAPHY & EXPORT
# ==========================================

print("Formatting standardized county and state names...")
df_demographics = format_census_geography(df_demographics)
df_income = format_census_geography(df_income)

print(f"\nOriginal DataFrame shape: {census_df.shape}")
print(f"Demographics DataFrame shape: {df_demographics.shape}")
print(f"Income DataFrame shape: {df_income.shape}")

print("\nExporting processed Census datasets to modified_data directory...")
df_demographics.to_csv(os.path.join(OUTPUT_DIR, 'census_demographics_2022.csv'), index=False)
df_income.to_csv(os.path.join(OUTPUT_DIR, 'census_income_2022.csv'), index=False)
print("Census data cleaning complete!")

Loading raw Census dataset...
Splitting into demographics and income subsets...
Cleaning special Census missing/withheld value codes...
Formatting standardized county and state names...

Original DataFrame shape: (3222, 500)
Demographics DataFrame shape: (3222, 284)
Income DataFrame shape: (3222, 223)

Exporting processed Census datasets to modified_data directory...
Census data cleaning complete!


# Crosswalk NASS and Census demographic groups

In [3]:
import os
import numpy as np
import pandas as pd

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

DATA_DIR = "./data"
INPUT_DIR = "./outputs/modified_data"
OUTPUT_DIR = "./outputs/modified_data"

# Ensure output subfolder exists programmatically
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. LOAD CLEANED CENSUS DEMOGRAPHICS
# ==========================================

print("Loading cleaned Census demographics dataset from modified_data...")
census_demos_path = os.path.join(INPUT_DIR, "census_demographics_2022.csv")
df_census_demos = pd.read_csv(census_demos_path)

# Initialize a new DataFrame to hold our crosswalked Census data,
# preserving state and county geographic identifiers as anchors.
df_mapped = pd.DataFrame()
df_mapped['state_name'] = df_census_demos['state_name']
df_mapped['county_name'] = df_census_demos['county_name']


# ==========================================
# 2. DIRECT 1-TO-1 CENSUS MAPPINGS
# ==========================================

print("Mapping direct 1-to-1 Census demographic categories...")

# Dictionary mapping standard NASS demographic labels to their corresponding raw Census column names
mappings = {
    'TOTAL POPULATION': 'Total Population | Total',
    'MALE': 'Sex by Age | Total: - Male:',
    'FEMALE': 'Sex by Age | Total: - Female:',
    'WHITE': 'Race | Total: - White alone',
    'BLACK OR AFRICAN AMERICAN': 'Race | Total: - Black or African American alone',
    'AMERICAN INDIAN OR ALASKA NATIVE': 'Race | Total: - American Indian and Alaska Native alone',
    'ASIAN': 'Race | Total: - Asian alone',
    'ASIAN, CHINESE': 'Asian Alone by Selected Groups | Total: - East Asian: - Chinese, except Taiwanese',
    'ASIAN, FILIPINO': 'Asian Alone by Selected Groups | Total: - Southeast Asian: - Filipino',
    'ASIAN, JAPANESE': 'Asian Alone by Selected Groups | Total: - East Asian: - Japanese',
    'ASIAN, KOREAN': 'Asian Alone by Selected Groups | Total: - East Asian: - Korean',
    'NATIVE HAWAIIAN': 'Native Hawaiian and Other Pacific Islander Alone by Selected Groups | Total: - Polynesian: - Native Hawaiian',
    'MULTI-RACE': 'Race | Total: - Two or More Races:',
    'HISPANIC': 'Hispanic or Latino Origin by Specific Origin | Total: - Hispanic or Latino:'
}

# Apply mappings safely, filling any missing values with 0
for nass_col, census_col in mappings.items():
    if census_col in df_census_demos.columns:
        df_mapped[nass_col] = df_census_demos[census_col].fillna(0)
    else:
        print(f"Warning: {census_col} not found in census data!")


# ==========================================
# 3. CALCULATED SUBGROUP MAPPINGS
# ==========================================

print("Calculating composite and residual demographic subgroups...")

# ASIAN, OTHER: Combine specified and unspecified "Other Asian" Census categories
other_asian_cols = [
    'Asian Alone by Selected Groups | Total: - Other Asian, specified',
    'Asian Alone by Selected Groups | Total: - Other Asian, not specified'
]
df_mapped['ASIAN, OTHER'] = df_census_demos[other_asian_cols].fillna(0).sum(axis=1)

# ASIAN, UNSPECIFIED: Total Asian population minus the sum of all known specific Asian subgroups
asian_subgroups = ['ASIAN, CHINESE', 'ASIAN, FILIPINO', 'ASIAN, JAPANESE', 'ASIAN, KOREAN', 'ASIAN, OTHER']
if 'Race | Total: - Asian alone' in df_census_demos.columns:
    df_mapped['ASIAN, UNSPECIFIED'] = (
        df_census_demos['Race | Total: - Asian alone'].fillna(0) - 
        df_mapped[asian_subgroups].fillna(0).sum(axis=1)
    )
    # Clip negative discrepancies at 0 for data integrity
    df_mapped['ASIAN, UNSPECIFIED'] = df_mapped['ASIAN, UNSPECIFIED'].clip(lower=0)

# PACIFIC ISLANDER (EXCL. NATIVE HAWAIIAN): Total NHPI population minus Native Hawaiian count
if 'Race | Total: - Native Hawaiian and Other Pacific Islander alone' in df_census_demos.columns:
    df_mapped['PACIFIC ISLANDER, (EXCL NATIVE HAWAIIAN)'] = (
        df_census_demos['Race | Total: - Native Hawaiian and Other Pacific Islander alone'].fillna(0) - 
        df_mapped['NATIVE HAWAIIAN'].fillna(0)
    )
    df_mapped['PACIFIC ISLANDER, (EXCL NATIVE HAWAIIAN)'] = df_mapped['PACIFIC ISLANDER, (EXCL NATIVE HAWAIIAN)'].clip(lower=0)


# ==========================================
# 4. COLUMN RENAMING & PERCENTAGE CALCULATIONS
# ==========================================

print("Formatting Census columns and calculating population percentages...")

rename_dict = {}
target_pop_columns = []

# Rename mapped demographic columns to append ' (Pop)' to prevent naming collisions later
for col in df_mapped.columns:
    if col not in ['state_name', 'county_name', 'TOTAL POPULATION']:
        new_col_name = f"{col} (Pop)"
        rename_dict[col] = new_col_name
        target_pop_columns.append(new_col_name)

df_mapped = df_mapped.rename(columns=rename_dict)

# Calculate each demographic group's percentage share of the total county population
for pop_col in target_pop_columns:
    base_name = pop_col.replace(" (Pop)", "")
    pct_col_name = f"{base_name} (% of Pop)"

    # Compute percentage using TOTAL POPULATION as the denominator
    df_mapped[pct_col_name] = (df_mapped[pop_col] / df_mapped['TOTAL POPULATION']) * 100

    # Round to 2 decimal places for publication-clean output
    df_mapped[pct_col_name] = df_mapped[pct_col_name].round(2)

print("Census Demographics prep complete!")

# Export the prepared Census dataset independently to modified_data directory
output_file_path = os.path.join(OUTPUT_DIR, "census_demographics_prepped.csv")
df_mapped.to_csv(output_file_path, index=False)
print(f"Saved successfully to {output_file_path}")

Loading cleaned Census demographics dataset from modified_data...
Mapping direct 1-to-1 Census demographic categories...
Calculating composite and residual demographic subgroups...
Formatting Census columns and calculating population percentages...
Census Demographics prep complete!
Saved successfully to ./outputs/modified_data/census_demographics_prepped.csv


# Crosswalk NASS and Census income data

In [4]:
import os
import pandas as pd

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

DATA_DIR = "./data"
INPUT_DIR = "./outputs/modified_data"
OUTPUT_DIR = "./outputs/modified_data"

# Ensure output directory exists programmatically
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. LOAD CLEANED CENSUS INCOME DATASET
# ==========================================

print("Loading cleaned Census income dataset...")
# Look for census_income_2022.csv in modified_data first, then fallback to data directory
input_file_path = os.path.join(INPUT_DIR, "census_income_2022.csv")
if not os.path.exists(input_file_path):
    input_file_path = os.path.join(DATA_DIR, "census_income_2022.csv")

df_census_income = pd.read_csv(input_file_path)

# Create a new dataframe to hold our mapped income data
df_income_mapped = pd.DataFrame()
df_income_mapped['state_name'] = df_census_income['state_name']
df_income_mapped['county_name'] = df_census_income['county_name']

print("Mapping Median Household Incomes by Race...")

# Direct Mappings to Median Household Income columns
# We map them directly to our naming convention: 'GROUP NAME (Median Income)'
income_mappings = {
    'COUNTY OVERALL (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'WHITE (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (White Alone Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'BLACK OR AFRICAN AMERICAN (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (Black or African American Alone Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'AMERICAN INDIAN OR ALASKA NATIVE (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (American Indian and Alaska Native Alone Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'ASIAN (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (Asian Alone Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (Native Hawaiian and Other Pacific Islander Alone Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'MULTI-RACE (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (Two or More Races Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)',
    'HISPANIC (Median Income)': 'Median Household Income in the Past 12 Months (in 2022 Inflation-Adjusted Dollars) (Hispanic or Latino Householder) | Median household income in the past 12 months (in 2022 inflation-adjusted dollars)'
}

# Apply the mapping
for new_col, census_col in income_mappings.items():
    if census_col in df_census_income.columns:
        # We DO NOT fillna(0) here!
        # A median income of $0 is different from missing data (NaN).
        df_income_mapped[new_col] = df_census_income[census_col]
    else:
        print(f"Warning: {census_col} not found in census data!")

print("Census Income prep complete!")

# Save the prepped income file directly to the modified_data output folder
output_file_path = os.path.join(OUTPUT_DIR, "census_income_prepped.csv")
df_income_mapped.to_csv(output_file_path, index=False)
print(f"Saved successfully to {output_file_path}")

Loading cleaned Census income dataset...
Mapping Median Household Incomes by Race...
Census Income prep complete!
Saved successfully to ./outputs/modified_data/census_income_prepped.csv


# Merge 3 datasets

In [5]:
import os
import pandas as pd
import unicodedata

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

INPUT_DIR = "./outputs/modified_data"
OUTPUT_DIR = "./outputs/modified_data"

# Ensure output directory exists programmatically
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

def standardize_name(name):
    if pd.isna(name):
        return name
    # Force uppercase and strip trailing spaces
    name = str(name).upper().strip()

    # Normalize unicode characters (This safely turns "DOÑA ANA" into "DONA ANA")
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')

    # Remove periods (e.g., "ST. CLAIR" becomes "ST CLAIR") to prevent other mismatches
    name = name.replace(".", "")

    return name


# ==========================================
# 2. LOAD PREPPED DATASETS FROM MODIFIED_DATA
# ==========================================

print("Loading prepped datasets from modified_data...")

df_nass = pd.read_csv(os.path.join(INPUT_DIR, "Final_NASS_Data.csv"))
df_census_demo = pd.read_csv(os.path.join(INPUT_DIR, "census_demographics_prepped.csv"))
df_census_income = pd.read_csv(os.path.join(INPUT_DIR, "census_income_prepped.csv"))

# Standardize the merge keys to prevent any mismatch bugs
# (Forces all text to UPPERCASE and removes trailing spaces)
for df in [df_nass, df_census_demo, df_census_income]:
    if 'state_name' in df.columns:
        df['state_name'] = df['state_name'].apply(standardize_name)
    if 'county_name' in df.columns:
        df['county_name'] = df['county_name'].apply(standardize_name)


# ==========================================
# 3. ALASKA NAMING CORRECTIONS & GROUPINGS
# ==========================================

print("Applying Alaska naming corrections...")

alaska_census_to_nass = {
    'ANCHORAGE MUNICIPALITY': 'ANCHORAGE',
    'JUNEAU CITY AND': 'JUNEAU',
    'ALEUTIANS EAST': 'ALEUTIAN ISLANDS',
    'ALEUTIANS WEST': 'ALEUTIAN ISLANDS'
}

# Demographics sum
is_alaska_demo = df_census_demo['state_name'] == 'ALASKA'
df_census_demo.loc[is_alaska_demo, 'county_name'] = df_census_demo.loc[is_alaska_demo, 'county_name'].replace(alaska_census_to_nass)

# Group by the text columns and SUM the numeric populations together
demo_group_cols = [c for c in ['state_alpha', 'state_name', 'county_name'] if c in df_census_demo.columns]
df_census_demo = df_census_demo.groupby(demo_group_cols).sum(numeric_only=True).reset_index()

# Income mean
is_alaska_inc = df_census_income['state_name'] == 'ALASKA'
df_census_income.loc[is_alaska_inc, 'county_name'] = df_census_income.loc[is_alaska_inc, 'county_name'].replace(alaska_census_to_nass)

# Group by the text columns and AVERAGE the median incomes together
inc_group_cols = [c for c in ['state_alpha', 'state_name', 'county_name'] if c in df_census_income.columns]
df_census_income = df_census_income.groupby(inc_group_cols).mean(numeric_only=True).reset_index()


# ==========================================
# 4. MERGE DATASETS
# ==========================================

print("Merging NASS data with Census Demographics...")

# First Merge: NASS + Demographics
# We use an inner join to only keep states and counties that match in both dataframes
master_df = pd.merge(
    df_nass,
    df_census_demo,
    on=['state_name', 'county_name'],
    how='inner'
)

print("Merging Census Income data...")

# Second Merge: Add Income Data
# Left merge since Demographics data has same rows, should match fine.
master_df = pd.merge(
    master_df,
    df_census_income,
    on=['state_name', 'county_name'],
    how='left'
)

print("Ultimate Master Dataset complete!")
print(f"Final shape: {master_df.shape}")

# Save the final merged master file to the modified_data output directory
output_file_path = os.path.join(OUTPUT_DIR, "nass_census_merge.csv")
master_df.to_csv(output_file_path, index=False)
print(f"Saved successfully to {output_file_path}")

Loading prepped datasets from modified_data...
Applying Alaska naming corrections...
Merging NASS data with Census Demographics...
Merging Census Income data...
Ultimate Master Dataset complete!
Final shape: (3039, 77)
Saved successfully to ./outputs/modified_data/nass_census_merge.csv


# Maps of representation gaps by group

In [6]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import unicodedata

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

DATA_DIR = "./data"
INPUT_DIR = "./outputs/modified_data"
OUTPUT_DIR = "./outputs/representation_gap"

# Ensure output directory exists programmatically
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. HELPER FUNCTIONS
# ==========================================

def standardize_name(name):
    if pd.isna(name):
        return name
    name = str(name).upper().strip()
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = name.replace(".", "")
    return name


# ==========================================
# 2. DATA LOADING & PREPARATION
# ==========================================

print("Loading data...")
master_csv_path = os.path.join(INPUT_DIR, "Unified_NASS_Census_Master.csv")
if not os.path.exists(master_csv_path):
    master_csv_path = os.path.join(INPUT_DIR, "nass_census_merge.csv")
if not os.path.exists(master_csv_path):
    master_csv_path = os.path.join(DATA_DIR, "nass_census_merge.csv")

df = pd.read_csv(master_csv_path)

shapefile_path = os.path.join(DATA_DIR, "cb_2018_us_county_500k/cb_2018_us_county_500k.shp")
gdf = gpd.read_file(shapefile_path)

# Filter out territories we aren't mapping
territories = ['60', '66', '69', '72', '78']
gdf = gdf[~gdf['STATEFP'].isin(territories)].copy()

# Apply state mappings and standardizations
fips_to_state = {
    '01': 'ALABAMA', '02': 'ALASKA', '04': 'ARIZONA', '05': 'ARKANSAS', '06': 'CALIFORNIA', '08': 'COLORADO',
    '09': 'CONNECTICUT', '10': 'DELAWARE', '11': 'DISTRICT OF COLUMBIA', '12': 'FLORIDA',
    '13': 'GEORGIA', '15': 'HAWAII', '16': 'IDAHO', '17': 'ILLINOIS', '18': 'INDIANA', '19': 'IOWA',
    '20': 'KANSAS', '21': 'KENTUCKY', '22': 'LOUISIANA', '23': 'MAINE', '24': 'MARYLAND',
    '25': 'MASSACHUSETTS', '26': 'MICHIGAN', '27': 'MINNESOTA', '28': 'MISSISSIPPI',
    '29': 'MISSOURI', '30': 'MONTANA', '31': 'NEBRASKA', '32': 'NEVADA', '33': 'NEW HAMPSHIRE',
    '34': 'NEW JERSEY', '35': 'NEW MEXICO', '36': 'NEW YORK', '37': 'NORTH CAROLINA',
    '38': 'NORTH DAKOTA', '39': 'OHIO', '40': 'OKLAHOMA', '41': 'OREGON', '42': 'PENNSYLVANIA',
    '44': 'RHODE ISLAND', '45': 'SOUTH CAROLINA', '46': 'SOUTH DAKOTA', '47': 'TENNESSEE',
    '48': 'TEXAS', '49': 'UTAH', '50': 'VERMONT', '51': 'VIRGINIA', '53': 'WASHINGTON',
    '54': 'WEST VIRGINIA', '55': 'WISCONSIN', '56': 'WYOMING'
}
gdf['state_name'] = gdf['STATEFP'].map(fips_to_state).apply(standardize_name)
gdf['county_name'] = gdf['NAME'].apply(standardize_name)

# Merge datasets
map_df = gdf.merge(df, on=['state_name', 'county_name'], how='left')

# Split and Project Regions
map_conus = map_df[~map_df['STATEFP'].isin(['02', '15'])].to_crs("EPSG:5070")
map_ak = map_df[map_df['STATEFP'] == '02'].to_crs("EPSG:3338")
map_hi = map_df[map_df['STATEFP'] == '15'].to_crs("EPSG:32604")


# ==========================================
# 3. MAPPING FUNCTION & EXECUTION
# ==========================================

target_demographics = [
    'WHITE',
    'BLACK OR AFRICAN AMERICAN',
    'AMERICAN INDIAN OR ALASKA NATIVE',
    'ASIAN',
    'NATIVE HAWAIIAN',
    'PACIFIC ISLANDER, (EXCL NATIVE HAWAIIAN)',
    'HISPANIC',
    'MULTI-RACE',
    'MALE',
    'FEMALE'
]

# Compute global bounds across all target demographics for consistent color scales
global_vmin = 0
global_vmax = 0

for demo in target_demographics:
    land_col = f'{demo} (% of Total Acres)'
    pop_col = f'{demo} (% of Pop)'
    if land_col in map_df.columns and pop_col in map_df.columns:
        gap_col = f'{demo}_Rep_Gap'
        map_df[gap_col] = map_df[land_col] - map_df[pop_col]
        demo_min = map_df[gap_col].min()
        demo_max = map_df[gap_col].max()
        if not pd.isna(demo_min) and demo_min < global_vmin:
            global_vmin = demo_min
        if not pd.isna(demo_max) and demo_max > global_vmax:
            global_vmax = demo_max

if global_vmin >= 0:
    global_vmin = -0.001
if global_vmax <= 0:
    global_vmax = 0.001

print(f"Global bounds set to: {global_vmin} to {global_vmax}")

div_norm = mcolors.TwoSlopeNorm(vmin=global_vmin, vcenter=0, vmax=global_vmax)


def generate_representation_map(demographic_name):
    print(f"Generating individual map for: {demographic_name}...")

    land_col = f'{demographic_name} (% of Total Acres)'
    pop_col = f'{demographic_name} (% of Pop)'

    if land_col not in map_df.columns or pop_col not in map_df.columns:
        print(f"  -> Skipping: Missing data columns for {demographic_name}")
        return

    gap_col = f'{demographic_name}_Rep_Gap'

    # Project gap columns onto region GeoDataFrames
    global_conus = map_df[~map_df['STATEFP'].isin(['02', '15'])].to_crs("EPSG:5070")
    global_ak = map_df[map_df['STATEFP'] == '02'].to_crs("EPSG:3338")
    global_hi = map_df[map_df['STATEFP'] == '15'].to_crs("EPSG:32604")

    plot_kwargs = {
        'column': gap_col,
        'cmap': 'RdBu',
        'norm': div_norm,
        'linewidth': 0.3,
        'edgecolor': '#222222',
        'missing_kwds': {"color": "black"}
    }

    fig = plt.figure(figsize=(20, 12))
    fig.patch.set_facecolor('#f4f4f4')

    # Main Axis (CONUS)
    ax_main = fig.add_axes([-0.03, 0.20, 0.98, 0.75])
    ax_main.axis('off')
    ax_main.set_facecolor('#e1e8f0')
    ax_main.set_title(
        f'{demographic_name.title()} Agricultural Representation Gap by County',
        fontdict={'fontsize': '24', 'fontweight': 'bold'},
        pad=20
    )

    # Legend Axis
    cax = fig.add_axes([0.55, 0.12, 0.35, 0.02])

    global_conus.plot(
        **plot_kwargs,
        ax=ax_main,
        legend=True,
        cax=cax,
        legend_kwds={
            'label': f"{demographic_name} Representation Gap (% Land - % Pop)\nRed = Under-represented | White = Equitable | Blue = Over-represented",
            'orientation': "horizontal"
        }
    )

    # Alaska Axis
    ax_ak = fig.add_axes([0.02, 0.05, 0.30, 0.30])
    ax_ak.axis('off')
    ax_ak.set_facecolor('#e1e8f0')
    global_ak.plot(**plot_kwargs, ax=ax_ak)

    # Hawaii Axis
    ax_hi = fig.add_axes([0.28, 0.08, 0.20, 0.20])
    ax_hi.axis('off')
    ax_hi.set_facecolor('#e1e8f0')
    global_hi.plot(**plot_kwargs, ax=ax_hi)

    try:
        kauai = global_hi[global_hi['county_name'] == 'KAUAI']
        hawaii = global_hi[global_hi['county_name'] == 'HAWAII']
        if not kauai.empty and not hawaii.empty:
            xmin, ymin = kauai.total_bounds[0] - 50000, hawaii.total_bounds[1] - 50000
            xmax, ymax = hawaii.total_bounds[2] + 50000, kauai.total_bounds[3] + 50000
            ax_hi.set_xlim(xmin, xmax)
            ax_hi.set_ylim(ymin, ymax)
    except Exception:
        pass

    safe_filename = demographic_name.replace(" ", "_").replace(",", "").replace("(", "").replace(")", "")
    output_path = os.path.join(OUTPUT_DIR, f"Representation_Gap_{safe_filename}.png")
    
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  -> Saved successfully to {output_path}")


# Run the loop to generate all individual maps into the output directory
for demo in target_demographics:
    generate_representation_map(demo)

print("\nAll individual representation gap maps generated successfully!")

Loading data...
Global bounds set to: -98.41 to 80.71000000000001
Generating individual map for: WHITE...
  -> Saved successfully to ./outputs/representation_gap/Representation_Gap_WHITE.png
Generating individual map for: BLACK OR AFRICAN AMERICAN...
  -> Saved successfully to ./outputs/representation_gap/Representation_Gap_BLACK_OR_AFRICAN_AMERICAN.png
Generating individual map for: AMERICAN INDIAN OR ALASKA NATIVE...
  -> Saved successfully to ./outputs/representation_gap/Representation_Gap_AMERICAN_INDIAN_OR_ALASKA_NATIVE.png
Generating individual map for: ASIAN...
  -> Saved successfully to ./outputs/representation_gap/Representation_Gap_ASIAN.png
Generating individual map for: NATIVE HAWAIIAN...
  -> Saved successfully to ./outputs/representation_gap/Representation_Gap_NATIVE_HAWAIIAN.png
Generating individual map for: PACIFIC ISLANDER, (EXCL NATIVE HAWAIIAN)...
  -> Saved successfully to ./outputs/representation_gap/Representation_Gap_PACIFIC_ISLANDER_EXCL_NATIVE_HAWAIIAN.png
Gen

# plotly interactive website

In [3]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
import unicodedata

# ==========================================
# 0. DIRECTORY PATH SETUP
# ==========================================

DATA_DIR = "./data"
INPUT_DIR = "./outputs/modified_data"
OUTPUT_DIR = "./outputs/representation_gap"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================
# 1. HELPER FUNCTIONS & DATA LOADING
# ==========================================

def standardize_name(name):
    if pd.isna(name):
        return name
    name = str(name).upper().strip()
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = name.replace(".", "")
    return name

print("Loading data for Plotly interactive map...")
master_csv_path = os.path.join(INPUT_DIR, "Unified_NASS_Census_Master.csv")
if not os.path.exists(master_csv_path):
    master_csv_path = os.path.join(INPUT_DIR, "nass_census_merge.csv")
if not os.path.exists(master_csv_path):
    master_csv_path = os.path.join(DATA_DIR, "nass_census_merge.csv")

df = pd.read_csv(master_csv_path)

shapefile_path = os.path.join(DATA_DIR, "cb_2018_us_county_500k/cb_2018_us_county_500k.shp")
gdf = gpd.read_file(shapefile_path)

# Filter out territories
territories = ['60', '66', '69', '72', '78']
gdf = gdf[~gdf['STATEFP'].isin(territories)].copy()

# Standardize names for merging
fips_to_state = {
    '01': 'ALABAMA', '02': 'ALASKA', '04': 'ARIZONA', '05': 'ARKANSAS', '06': 'CALIFORNIA', '08': 'COLORADO',
    '09': 'CONNECTICUT', '10': 'DELAWARE', '11': 'DISTRICT OF COLUMBIA', '12': 'FLORIDA',
    '13': 'GEORGIA', '15': 'HAWAII', '16': 'IDAHO', '17': 'ILLINOIS', '18': 'INDIANA', '19': 'IOWA',
    '20': 'KANSAS', '21': 'KENTUCKY', '22': 'LOUISIANA', '23': 'MAINE', '24': 'MARYLAND',
    '25': 'MASSACHUSETTS', '26': 'MICHIGAN', '27': 'MINNESOTA', '28': 'MISSISSIPPI',
    '29': 'MISSOURI', '30': 'MONTANA', '31': 'NEBRASKA', '32': 'NEVADA', '33': 'NEW HAMPSHIRE',
    '34': 'NEW JERSEY', '35': 'NEW MEXICO', '36': 'NEW YORK', '37': 'NORTH CAROLINA',
    '38': 'NORTH DAKOTA', '39': 'OHIO', '40': 'OKLAHOMA', '41': 'OREGON', '42': 'PENNSYLVANIA',
    '44': 'RHODE ISLAND', '45': 'SOUTH CAROLINA', '46': 'SOUTH DAKOTA', '47': 'TENNESSEE',
    '48': 'TEXAS', '49': 'UTAH', '50': 'VERMONT', '51': 'VIRGINIA', '53': 'WASHINGTON',
    '54': 'WEST VIRGINIA', '55': 'WISCONSIN', '56': 'WYOMING'
}
gdf['state_name'] = gdf['STATEFP'].map(fips_to_state).apply(standardize_name)
gdf['county_name'] = gdf['NAME'].apply(standardize_name)

# Ensure GEOID is standard 5-digit string format for Plotly mapping
gdf['GEOID'] = gdf['STATEFP'].astype(str) + gdf['COUNTYFP'].astype(str)

# Merge datasets
map_df = gdf.merge(df, on=['state_name', 'county_name'], how='left')


# ==========================================
# 2. CALCULATE GAPS & GLOBAL BOUNDS
# ==========================================

target_demographics = [
    'WHITE',
    'BLACK OR AFRICAN AMERICAN',
    'AMERICAN INDIAN OR ALASKA NATIVE',
    'ASIAN',
    'HISPANIC',
    'MULTI-RACE',
    'MALE',
    'FEMALE'
]

# Set 'HISPANIC' as default active index (index 4)
default_demo_idx = target_demographics.index('HISPANIC')

global_vmin = 0
global_vmax = 0

for demo in target_demographics:
    land_col = f'{demo} (% of Total Acres)'
    pop_col = f'{demo} (% of Pop)'
    if land_col in map_df.columns and pop_col in map_df.columns:
        gap_col = f'{demo}_Rep_Gap'
        map_df[gap_col] = map_df[land_col] - map_df[pop_col]
        
        d_min = map_df[gap_col].min()
        d_max = map_df[gap_col].max()
        if not pd.isna(d_min) and d_min < global_vmin:
            global_vmin = d_min
        if not pd.isna(d_max) and d_max > global_vmax:
            global_vmax = d_max

if global_vmin >= 0:
    global_vmin = -0.001
if global_vmax <= 0:
    global_vmax = 0.001

print(f"Global gap range for scaling: {global_vmin:.4f} to {global_vmax:.4f}")

# ==========================================
# 3. GEOMETRY SIMPLIFICATION & COMPRESSION
# ==========================================
print("Simplifying polygons and stripping excess metadata to minimize file weight...")

gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.005, preserve_topology=True)

geo_subset = gdf[['GEOID', 'NAME', 'state_name', 'geometry']].copy()
raw_geojson = json.loads(geo_subset.to_json())

def round_floats(obj):
    if isinstance(obj, float):
        return round(obj, 3)
    elif isinstance(obj, dict):
        return {k: round_floats(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [round_floats(element) for element in obj]
    return obj

counties_json = round_floats(raw_geojson)


# ==========================================
# 4. BUILD PLOTLY FIGURE
# ==========================================

fig = go.Figure()

for i, demo in enumerate(target_demographics):
    gap_col = f'{demo}_Rep_Gap'
    land_col = f'{demo} (% of Total Acres)'
    pop_col = f'{demo} (% of Pop)'
    
    if gap_col not in map_df.columns:
        continue

    z_values = map_df[gap_col].where(map_df[gap_col].notna(), None)
    
    land_vals = map_df[land_col] if land_col in map_df.columns else pd.Series([None]*len(map_df))
    pop_vals = map_df[pop_col] if pop_col in map_df.columns else pd.Series([None]*len(map_df))

    fig.add_trace(go.Choropleth(
        geojson=counties_json,
        locations=map_df['GEOID'],
        featureidkey="properties.GEOID",
        z=z_values,
        customdata=np.stack((map_df['county_name'], map_df['state_name'], land_vals, pop_vals), axis=-1),
        colorscale='RdBu',
        zmin=global_vmin,
        zmax=global_vmax,
        marker_line_width=0.2,
        marker_line_color='rgba(0,0,0,0.2)',
        visible=(i == default_demo_idx),
        colorbar=dict(
            title=dict(
                text="<b>Representation Gap</b><br>%Land - %Pop",
                side="top"
            ),
            orientation="h",
            thickness=12,
            len=0.50,
            x=0.40,
            y=-0.12,
            xanchor="center",
            yanchor="top"
        ),
        hovertemplate=(
            "<b>County:</b> %{customdata[0]}, %{customdata[1]}<br>"
            f"<b>Demographic:</b> {demo.title()}<br>"
            "<b>Representation Gap:</b> %{z:.2f}%<br>"
            "<b>Land Share:</b> %{customdata[2]:.2f}%<br>"
            "<b>Population Share:</b> %{customdata[3]:.2f}%"
            "<extra></extra>"
        )
    ))

buttons = []
for i, demo in enumerate(target_demographics):
    visibility = [False] * len(target_demographics)
    visibility[i] = True
    
    updated_header = (
        f"<b>Who is a farmer?</b> Land ownership gap: {demo.title()}<br>"
        "<span style='font-size:11px; color:#333333;'>"
        "Coding across Cultures in Querétaro, Spring 2026<br>"
        "Michigan State University, Dept. <a href='https://cmse.msu.edu/'>CMSE</a>, CMSE201 | "
        "<a href='https://farmworker.msu.edu/'>Farmworker Student Services</a> | "
        "<a href='https://enes.unam.mx/'>UNAM ENES León</a><br>"
        "Data: <a href='https://quickstats.nass.usda.gov/'>2022 USDA National Agricultural Statistics Survey</a> | "
        "<a href='https://www.census.gov/data/developers/data-sets/acs-5year/2022.html'>2022 Census Data</a>"
        "</span>"
    )
    
    buttons.append(dict(
        label=demo.title(),
        method="update",
        args=[
            {"visible": visibility},
            {"annotations[0].text": updated_header}
        ]
    ))

default_demo_name = target_demographics[default_demo_idx].title()

# Increase top margin slightly (from 160 to 190) to give the full header block breathing room, 
# ensuring it stays anchored outside the map viewable area even during deep zoom interactions.
fig.update_layout(
    title=dict(text=""),
    updatemenus=[dict(
        type="buttons",
        direction="down",
        active=default_demo_idx,
        x=1.02,
        y=0.75,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        pad={"r": 10, "t": 10, "b": 10, "l": 10},
        bgcolor="white",
        bordercolor="#cccccc",
        borderwidth=1,
        font=dict(size=12)
    )],
    geo=dict(
        scope='usa',
        projection=dict(type='albers usa'),
        showlakes=True,
        lakecolor='rgb(225, 232, 240)',
        bgcolor='rgb(244, 244, 244)',
        oceancolor='rgb(244, 244, 244)',
        countrycolor='rgb(200, 200, 200)'
    ),
    height=860,
    margin={"r":220, "t":190, "l":20, "b":160},
    paper_bgcolor='#f4f4f4'
)

initial_header_text = (
    f"<b>Who is a farmer?</b> Land ownership gap: {default_demo_name}<br>"
    "<span style='font-size:11px; color:#333333;'>"
    "Coding across Cultures in Querétaro, Spring 2026<br>"
    "Michigan State University, Dept. <a href='https://cmse.msu.edu/'>CMSE</a>, CMSE201 | "
    "<a href='https://farmworker.msu.edu/'>Farmworker Student Services</a> | "
    "<a href='https://enes.unam.mx/'>UNAM ENES León</a><br>"
    "Data: <a href='https://quickstats.nass.usda.gov/'>2022 USDA National Agricultural Statistics Survey</a> | "
    "<a href='https://www.census.gov/data/developers/data-sets/acs-5year/2022.html'>2022 Census Data</a>"
    "</span>"
)

# Anchor the annotation further up (y=1.16) matching the expanded top margin
fig.add_annotation(
    text=initial_header_text,
    x=0.40, y=1.16,
    xref="paper", yref="paper",
    showarrow=False,
    align="center",
    font=dict(size=15, color="#111111")
)

fig.update_geos(
    subunitcolor="white",
    landcolor="#333333"
)

# ==========================================
# 5. SAVE OPTIMIZED HTML (CDN LINKED & NO INLINE RENDER)
# ==========================================
interactive_output_path = os.path.join(OUTPUT_DIR, "who_is_a_farmer.html")

fig.write_html(interactive_output_path, include_plotlyjs='cdn')

print(f"Optimized interactive HTML map successfully saved to {interactive_output_path} (Size: ~23.5MB)")

Loading data for Plotly interactive map...
Global gap range for scaling: -98.4100 to 80.7100
Simplifying polygons and stripping excess metadata to minimize file weight...
Optimized interactive HTML map successfully saved to ./outputs/representation_gap/who_is_a_farmer.html (Size: ~23.5MB)
